# Hadoop Streaming Lab: Doanh thu theo nhóm sản phẩm (Colab)

Chạy Hadoop 3.3.6 chế độ một máy (pseudo-distributed) trong Colab, đủ các lệnh `hadoop fs`, `hadoop jar`, `hadoop-streaming`.

**Lưu ý**
- Notebook này **chưa có output**: hãy chạy lần lượt từng cell rồi **chụp màn hình ngay** sau mỗi bước làm bằng chứng nộp bài.
- Runtime Colab bị ngắt thì mất toàn bộ HDFS và cài đặt; phải chạy lại từ Cell 1.
- Kết quả đúng (đã kiểm chứng cục bộ bằng pipe Unix): Electronics 3703769311, Home 410315065, Fashion 218597763 là Top 3.
- Hỏi giảng viên xem Colab có được chấp nhận thay cho LabEx không.

**Trước khi chạy:** sửa `MSSV` và `HOTEN` ở Cell 2.

## 0. Cài đặt môi trường

### Cell 1: Cài Java 8 và tải Hadoop

In [1]:
!apt-get -qq install -y openjdk-8-jdk-headless > /dev/null

# Xóa file tải dở và thư mục giải nén dở từ lần trước
# !rm -rf /content/hadoop-3.3.6 /content/hadoop-3.3.6.tar.gz

# Tải lại (-c: tiếp tục nếu đứt giữa chừng; hiện tiến độ gọn)
!apt-get -qq install -y aria2 > /dev/null
!aria2c -x 8 -s 8 -c -d /content https://archive.apache.org/dist/hadoop/common/hadoop-3.3.6/hadoop-3.3.6.tar.gz
# !wget -c --progress=dot:giga https://archive.apache.org/dist/hadoop/common/hadoop-3.3.6/hadoop-3.3.6.tar.gz

# Kiểm tra file nguyên vẹn trước khi giải nén
!ls -lh /content/hadoop-3.3.6.tar.gz
!gzip -t /content/hadoop-3.3.6.tar.gz && echo "File OK"

!tar -xzf /content/hadoop-3.3.6.tar.gz -C /content
!ls /content/hadoop-3.3.6


10/07 08:17:37 [NOTICE] Downloading 1 item(s)
 *** Download Progress Summary as of Wed Oct  7 08:18:38 2026 *** 
=
[#11429f 29MiB/696MiB(4%) CN:8 DL:485KiB ETA:23m27s]
FILE: /content/hadoop-3.3.6.tar.gz
-

 *** Download Progress Summary as of Wed Oct  7 08:19:39 2026 *** 
=
[#11429f 58MiB/696MiB(8%) CN:8 DL:569KiB ETA:19m5s]
FILE: /content/hadoop-3.3.6.tar.gz
-

 *** Download Progress Summary as of Wed Oct  7 08:20:39 2026 *** 
=
[#11429f 95MiB/696MiB(13%) CN:8 DL:593KiB ETA:17m16s]
FILE: /content/hadoop-3.3.6.tar.gz
-

 *** Download Progress Summary as of Wed Oct  7 08:21:40 2026 *** 
=
[#11429f 128MiB/696MiB(18%) CN:8 DL:631KiB ETA:15m19s]
FILE: /content/hadoop-3.3.6.tar.gz
-

 *** Download Progress Summary as of Wed Oct  7 08:22:40 2026 *** 
=
[#11429f 162MiB/696MiB(23%) CN:8 DL:625KiB ETA:14m33s]
FILE: /content/hadoop-3.3.6.tar.gz
-

 *** Download Progress Summary as of Wed Oct  7 08:23:41 2026 *** 
=
[#11429f 200MiB/696MiB(28%) CN:8 DL:568KiB ETA:14m53s]
FILE: /content/hadoop-3.3

### Cell 2: Biến môi trường (sửa MSSV, HOTEN)

In [2]:
import os
MSSV = "A47179"        # <-- thay bằng MSSV của bạn
HOTEN = "PhamDaiNghia"       # <-- thay bằng họ tên (không dấu, viết liền)

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["HADOOP_HOME"] = "/content/hadoop-3.3.6"
os.environ["PATH"] += ":/usr/lib/jvm/java-8-openjdk-amd64/bin:/content/hadoop-3.3.6/bin:/content/hadoop-3.3.6/sbin"
os.environ["MSSV"] = MSSV
for k in ["HDFS_NAMENODE_USER", "HDFS_DATANODE_USER", "HDFS_SECONDARYNAMENODE_USER",
          "YARN_RESOURCEMANAGER_USER", "YARN_NODEMANAGER_USER"]:
    os.environ[k] = "root"   # Colab chạy bằng root; Hadoop 3 yêu cầu khai báo

!hadoop version | head -n 2

Hadoop 3.3.6
Source code repository https://github.com/apache/hadoop.git -r 1be78238728da9266a4f88195058f08fd012bf9c


### Cell 3: Cấu hình HDFS

In [3]:
%%bash
cd $HADOOP_HOME/etc/hadoop
cat > core-site.xml <<'EOF'
<configuration>
  <property><name>fs.defaultFS</name><value>hdfs://localhost:9000</value></property>
</configuration>
EOF
cat > hdfs-site.xml <<'EOF'
<configuration>
  <property><name>dfs.replication</name><value>1</value></property>
  <property><name>dfs.namenode.name.dir</name><value>file:///content/hdfs/name</value></property>
  <property><name>dfs.datanode.data.dir</name><value>file:///content/hdfs/data</value></property>
</configuration>
EOF
echo "export JAVA_HOME=$JAVA_HOME" >> hadoop-env.sh
echo "Đã ghi cấu hình"

Đã ghi cấu hình


### Cell 4: Format và khởi động HDFS (đúng khi `jps` thấy NameNode, DataNode và `Live datanodes (1)`)

In [4]:
!hdfs namenode -format -force -nonInteractive > /content/format_log.txt 2>&1; tail -n 3 /content/format_log.txt
!hdfs --daemon start namenode
!hdfs --daemon start datanode
!sleep 15
!jps
!hdfs dfsadmin -report | head -n 15

/************************************************************
SHUTDOWN_MSG: Shutting down NameNode at 2b63c8a3c4cc/172.28.0.12
************************************************************/
[0.024s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.024s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
8387 NameNode
8627 Jps
8447 DataNode
Configured Capacity: 115658190848 (107.72 GB)
Present Capacity: 91157602304 (84.90 GB)
DFS Remaining: 91157577728 (84.90 GB)
DFS Used: 24576 (24 KB)
DFS Used%: 0.00%
Replicated Blocks:
	Under replicated blocks: 0
	Blocks with corrupt replicas: 0
	Missing blocks: 0
	Missing blocks (with replication factor 1): 0
	Low redundancy blocks with highest priority to recover: 0
	Pending deletion blocks: 0
Erasure Coded Block Groups: 
	Low redundancy b

### Cell 5: Upload 3 file từ máy (orders_2026_09.csv, mapper_revenue.py, reducer_revenue.py)

In [5]:
from google.colab import files
files.upload()
!chmod +x mapper_revenue.py reducer_revenue.py
!ls -la /content/*.csv /content/*.py

Saving mapper_revenue.py to mapper_revenue.py
Saving orders_2026_09.csv to orders_2026_09.csv
Saving reducer_revenue.py to reducer_revenue.py
-rwxr-xr-x 1 root root    241 Oct  7 08:42 /content/mapper_revenue.py
-rw-r--r-- 1 root root 121497 Oct  7 08:42 /content/orders_2026_09.csv
-rwxr-xr-x 1 root root    432 Oct  7 08:42 /content/reducer_revenue.py


## Bước 1: Đưa dữ liệu lên HDFS  (chụp ảnh cả cell này)

In [6]:
!hadoop fs -mkdir -p /user/{MSSV}/orders
!hadoop fs -put -f orders_2026_09.csv /user/{MSSV}/orders/
!hadoop fs -ls -h /user/{MSSV}/orders
!hadoop fs -head /user/{MSSV}/orders/orders_2026_09.csv

Found 1 items
-rw-r--r--   1 root supergroup    118.6 K 2026-10-07 08:42 /user/A47179/orders/orders_2026_09.csv
transaction_id,order_date,order_time,province,category,channel,quantity,unit_price,amount,status,shipping_days
TX000001,2026-09-30,02:25:00,Ha Noi,Beauty,web,1,294947,294947,SUCCESS,2
TX000002,2026-09-23,23:58:00,Da Nang,Grocery,mobile,1,265514,265514,SUCCESS,3
TX000003,2026-09-05,15:38:00,Ha Noi,Electronics,partner,1,12741238,12741238,SUCCESS,3
TX000004,2026-09-09,18:01:00,Hai Phong,Electronics,mobile,1,8846997,8846997,SUCCESS,4
TX000005,2026-09-10,12:50:00,Hai Phong,Beauty,web,1,593405,593405,RETURNED,6
TX000006,2026-09-11,10:10:00,Dong Nai,Home,web,1,2839709,2839709,FAILED,5
TX000007,2026-09-13,01:19:00,Ha Noi,Beauty,web,1,839161,839161,SUCCESS,1
TX000008,2026-09-29,13:20:00,Ha Noi,Beauty,partner,2,1061571,2123142,SUCCESS,1
TX000009,2026-09-25,10:30:00,Binh Duong,Beauty,partner,2,411680,823360,SUCCESS,3
TX000010,2026-09-05,20:13:00,Dong Nai,Fashion,web,1,440527,440527,SUCC

## Bước 2: Test mapper/reducer cục bộ trước

`sort` trong pipe đóng vai trò shuffle & sort của Hadoop. Kết quả phải ra 8 dòng khớp số liệu ở phần đầu.

In [7]:
%%bash
cd /content
echo "Số cặp key-value mapper xuất ra (kỳ vọng 1240):"
cat orders_2026_09.csv | ./mapper_revenue.py | wc -l
echo "--- Kết quả reducer ---"
cat orders_2026_09.csv | ./mapper_revenue.py | sort -k1,1 | ./reducer_revenue.py

Số cặp key-value mapper xuất ra (kỳ vọng 1240):
1240
--- Kết quả reducer ---
Beauty	111032003
Book	39306022
Electronics	3703769311
Fashion	218597763
Grocery	80983147
Home	410315065
Sport	166194354
Toy	83881779


## (Tùy chọn) Bật YARN: chạy TRƯỚC Bước 3 nếu muốn job chạy trên YARN

Nếu bỏ qua, job chạy bằng LocalJobRunner (mã `job_local...`), không có `application_...`. Nếu YARN lỗi, dừng nó và xóa `mapred-site.xml`, kết quả vẫn như nhau.

In [8]:
%%bash
cd $HADOOP_HOME/etc/hadoop
cat > mapred-site.xml <<'EOF'
<configuration>
  <property><name>mapreduce.framework.name</name><value>yarn</value></property>
  <property><name>yarn.app.mapreduce.am.env</name><value>HADOOP_MAPRED_HOME=/content/hadoop-3.3.6</value></property>
  <property><name>mapreduce.map.env</name><value>HADOOP_MAPRED_HOME=/content/hadoop-3.3.6</value></property>
  <property><name>mapreduce.reduce.env</name><value>HADOOP_MAPRED_HOME=/content/hadoop-3.3.6</value></property>
</configuration>
EOF
cat > yarn-site.xml <<'EOF'
<configuration>
  <property><name>yarn.nodemanager.aux-services</name><value>mapreduce_shuffle</value></property>
  <property><name>yarn.nodemanager.vmem-check-enabled</name><value>false</value></property>
</configuration>
EOF
yarn --daemon start resourcemanager
yarn --daemon start nodemanager
sleep 20
jps

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
8387 NameNode
9270 NodeManager
9657 Jps
9211 ResourceManager
8447 DataNode


## Bước 3: Chạy Hadoop Streaming  (chụp ảnh lệnh và phần cuối log)

Cuối log cần thấy `completed successfully` và bộ đếm gần như: `Map input records=1501`, `Map output records=1240`, `Reduce input groups=8`.

In [10]:
%%bash
cd /content
hadoop fs -rm -r -f /user/$MSSV/output/revenue_by_category
hadoop jar $HADOOP_HOME/share/hadoop/tools/lib/hadoop-streaming-3.3.6.jar \
  -D mapreduce.input.fileinputformat.split.minsize=134217728 \
  -files /content/mapper_revenue.py,/content/reducer_revenue.py \
  -mapper mapper_revenue.py \
  -reducer reducer_revenue.py \
  -input  /user/$MSSV/orders/orders_2026_09.csv \
  -output /user/$MSSV/output/revenue_by_category 2>&1 | tee /content/streaming_log.txt

Deleted /user/A47179/output/revenue_by_category
packageJobJar: [/tmp/hadoop-unjar1051910837452292286/] [] /tmp/streamjob5628242305541414577.jar tmpDir=null
2026-10-07 08:47:14,768 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at /0.0.0.0:8032
2026-10-07 08:47:15,161 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at /0.0.0.0:8032
2026-10-07 08:47:15,544 INFO mapreduce.JobResourceUploader: Disabling Erasure Coding for path: /tmp/hadoop-yarn/staging/root/.staging/job_1791362574647_0002
2026-10-07 08:47:16,600 INFO mapred.FileInputFormat: Total input files to process : 1
2026-10-07 08:47:17,117 INFO mapreduce.JobSubmitter: number of splits:1
2026-10-07 08:47:17,765 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_1791362574647_0002
2026-10-07 08:47:17,765 INFO mapreduce.JobSubmitter: Executing with tokens: []
2026-10-07 08:47:18,117 INFO conf.Configuration: resource-types.xml not found
2026-10-07 08:47:18,118 INFO re

In [11]:
# Xem nhanh các bộ đếm chính trong log để chụp ảnh
!grep -E "number of splits|completed successfully|Launched (map|reduce) tasks|Map input records|Map output records|Reduce input groups|Reduce output records" /content/streaming_log.txt

2026-10-07 08:47:17,117 INFO mapreduce.JobSubmitter: number of splits:1
2026-10-07 08:47:45,097 INFO mapreduce.Job: Job job_1791362574647_0002 completed successfully
		Launched map tasks=1
		Launched reduce tasks=1
		Map input records=1501
		Map output records=1240
		Reduce input groups=8
		Reduce output records=8


## Bước 4: Kiểm tra và tải kết quả  (chụp ảnh cả cell này)

In [12]:
%%bash
cd /content
OUT=/user/$MSSV/output/revenue_by_category
echo "== ls =="; hadoop fs -ls $OUT
echo "== part-00000 (xếp theo key) =="; hadoop fs -cat $OUT/part-00000
echo "== Top 3 theo doanh thu =="; hadoop fs -cat $OUT/part-00000 | sort -k2,2nr | head -3
hadoop fs -get -f $OUT/part-00000 /content/part-00000
echo "Đã tải part-00000 về /content"

== ls ==
Found 2 items
-rw-r--r--   1 root supergroup          0 2026-10-07 08:47 /user/A47179/output/revenue_by_category/_SUCCESS
-rw-r--r--   1 root supergroup        133 2026-10-07 08:47 /user/A47179/output/revenue_by_category/part-00000
== part-00000 (xếp theo key) ==
Beauty	111032003
Book	39306022
Electronics	3703769311
Fashion	218597763
Grocery	80983147
Home	410315065
Sport	166194354
Toy	83881779
== Top 3 theo doanh thu ==
Electronics	3703769311
Home	410315065
Fashion	218597763
Đã tải part-00000 về /content


## Bước 5: Quan sát job và liên hệ YARN

- Nếu đã bật YARN: chạy cell dưới và chụp ảnh (có ID `application_...` và trạng thái `FINISHED`/`SUCCEEDED`).
- Nếu chạy chế độ local: chụp phần log job và ghi chú trong báo cáo rằng Colab chạy LocalJobRunner.

In [13]:
!yarn application -list -appStates ALL

2026-10-07 08:49:41,225 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at /0.0.0.0:8032
Total number of applications (application-types: [], states: [NEW, NEW_SAVING, SUBMITTED, ACCEPTED, RUNNING, FINISHED, FAILED, KILLED] and tags: []):2
                Application-Id	    Application-Name	    Application-Type	      User	     Queue	             State	       Final-State	       Progress	                       Tracking-URL
application_1791362574647_0002	streamjob5628242305541414577.jar	           MAPREDUCE	      root	   default	          FINISHED	         SUCCEEDED	           100%	http://2b63c8a3c4cc:19888/jobhistory/job/job_1791362574647_0002
application_1791362574647_0001	streamjob5303481666342711259.jar	           MAPREDUCE	      root	   default	          FINISHED	         SUCCEEDED	           100%	http://2b63c8a3c4cc:19888/jobhistory/job/job_1791362574647_0001


## Đóng gói bài nộp

Ảnh chụp màn hình và báo cáo PDF bạn tự thêm vào thư mục sau khi tải về.

In [14]:
import os
FOLDER = f"{MSSV}_{HOTEN}_HadoopLab"
os.makedirs(f"/content/{FOLDER}", exist_ok=True)
!cp /content/part-00000 /content/streaming_log.txt /content/mapper_revenue.py /content/reducer_revenue.py /content/{FOLDER}/
!cd /content && zip -r {FOLDER}.zip {FOLDER}
from google.colab import files
files.download(f"/content/{FOLDER}.zip")

  adding: A47179_PhamDaiNghia_HadoopLab/ (stored 0%)
  adding: A47179_PhamDaiNghia_HadoopLab/reducer_revenue.py (deflated 56%)
  adding: A47179_PhamDaiNghia_HadoopLab/streaming_log.txt (deflated 65%)
  adding: A47179_PhamDaiNghia_HadoopLab/part-00000 (deflated 15%)
  adding: A47179_PhamDaiNghia_HadoopLab/mapper_revenue.py (deflated 31%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>